# Analysis: Feature Tables

Creates tables for `logreg`, `xgboost`, `catboost` 


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display
from pandas.io.formats.style import Styler

MODEL_CONFIG = {
    'logreg': {
        'label': 'LogReg',
        'prefix': 'logreg_results',
        'baseline_feature_battery': 'extended_only',
    },
    'xgboost': {
        'label': 'XGBoost',
        'prefix': 'xgboost_results',
        'baseline_feature_battery': 'extended_only',
    },
    'catboost': {
        'label': 'CatBoost',
        'prefix': 'catboost_results',
        'baseline_feature_battery': 'extended_only',
    },
}


def base_dataset_from(dataset, label):
    suf = '_' + str(label)
    if isinstance(dataset, str) and dataset.endswith(suf):
        return dataset[: -len(suf)]
    return dataset

def preprocess_results(df: pd.DataFrame, strip_baseline_suffix: bool = False) -> pd.DataFrame:
    if df.empty:
        return df
    req = {'dataset', 'label', 'feature_battery', 'task_type', 'error', 'test_f1', 'test_r2'}
    missing = req - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    df_ok = df[df['error'].isna()].copy()
    df_ok['test_f1'] = pd.to_numeric(df_ok.get('test_f1'), errors='coerce')
    df_ok['test_r2'] = pd.to_numeric(df_ok.get('test_r2'), errors='coerce')

    if strip_baseline_suffix:
        df_ok['feature_battery'] = (
            df_ok['feature_battery']
            .astype(str)
            .str.replace('+baseline_state_and_edge', '', regex=False)
        )

    df_ok['base_dataset'] = [base_dataset_from(d, l) for d, l in zip(df_ok['dataset'], df_ok['label'])]
    df_ok['_score'] = np.where(
        df_ok['task_type'] == 'classification', df_ok['test_f1'],
        np.where(df_ok['task_type'] == 'regression', df_ok['test_r2'], np.nan),
    )
    df_ok = df_ok.sort_values('_score', ascending=False).drop_duplicates(
        subset=['base_dataset', 'label', 'task_type', 'feature_battery'],
        keep='first',
    ).drop(columns=['_score'])
    return df_ok

def load_all_csvs_in_dir(d: Path) -> pd.DataFrame:
    if not d.exists() or not d.is_dir():
        return pd.DataFrame()
    dfs = [pd.read_csv(f) for f in sorted(d.glob('*.csv'))]
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

def load_mode_group_df(model_key: str) -> tuple[pd.DataFrame, str]:
    cfg = MODEL_CONFIG[model_key]

    results_root = Path('results')
    model_folder = model_key

    base_dir = results_root / model_folder / 'extended_only'
    base = load_all_csvs_in_dir(base_dir)
    if base.empty:
        return pd.DataFrame(), cfg['baseline_feature_battery']
    uniq = sorted(x for x in base['feature_battery'].dropna().unique())
    baseline_feature_battery = uniq[0] if uniq else cfg['baseline_feature_battery']

    mode_dirs = [
        results_root / model_folder / 'new_features',
        results_root / model_folder / 'new_features_plus_extended',
        results_root / model_folder / 'new_features_plus_extended_shannon_entropies',
    ]

    frames = []
    for d in mode_dirs:
        mode_df = load_all_csvs_in_dir(d)
        if mode_df.empty:
            continue
        combined = pd.concat([base, mode_df], ignore_index=True)
        df_ok = preprocess_results(combined, strip_baseline_suffix=False)
        if not df_ok.empty:
            frames.append(df_ok)

    if not frames:
        return pd.DataFrame(), baseline_feature_battery

    combined = pd.concat(frames, ignore_index=True)
    combined['_metric'] = np.where(combined['task_type'].eq('classification'), combined['test_f1'], combined['test_r2'])
    dedup_cols = [c for c in ['base_dataset', 'label', 'task_type', 'test_type', 'test_subject_id', 'feature_battery', 'test_n_classes'] if c in combined.columns]
    combined = combined.sort_values('_metric').drop_duplicates(subset=dedup_cols, keep='last').drop(columns=['_metric'])
    return combined, baseline_feature_battery

def style_vs_baseline(tbl: pd.DataFrame, baseline_col: str):
    if baseline_col not in tbl.columns:
        print(f"[SKIP] baseline column {baseline_col!r} not found in table columns.")
        return
    tbl_num = tbl.copy().apply(pd.to_numeric, errors='coerce')
    tbl_num = tbl_num[tbl_num[baseline_col].notna()].copy()
    if tbl_num.empty:
        print(f"No rows to display after baseline alignment: {baseline_col}")
        return

    mean_vals = tbl_num.mean(axis=0, skipna=True)
    mean_adv = mean_vals - mean_vals[baseline_col]
    baseline_mean = mean_vals[baseline_col]
    baseline_mean_abs = abs(baseline_mean)
    mean_adv_pct = (mean_adv / baseline_mean_abs * 100) if (pd.notna(baseline_mean) and baseline_mean != 0) else mean_adv * np.nan

    if isinstance(tbl_num.index, pd.MultiIndex):
        mean_idx = ('MEAN', 'mean_label')
        mean_adv_idx = ('MEAN_ADV_VS_BASELINE', 'mean_label')
        summary_idx = pd.MultiIndex.from_tuples([mean_idx, mean_adv_idx], names=tbl_num.index.names)
    else:
        mean_idx = 'MEAN'
        mean_adv_idx = 'MEAN_ADV_VS_BASELINE'
        summary_idx = pd.Index([mean_idx, mean_adv_idx], name=tbl_num.index.name)

    summary_tbl = pd.DataFrame([mean_vals.values, mean_adv_pct.values], index=summary_idx, columns=tbl_num.columns)
    tbl_num = pd.concat([tbl_num, summary_tbl], axis=0)

    disp = pd.DataFrame('', index=tbl_num.index, columns=tbl_num.columns)
    base_vals = tbl_num[baseline_col]
    for col in tbl_num.columns:
        col_vals = tbl_num[col]
        if col == baseline_col:
            disp[col] = col_vals.map(lambda v: f"{v:.3f}" if pd.notna(v) else '')
            continue
        out = []
        for idx, v, b in zip(tbl_num.index, col_vals, base_vals):
            if pd.isna(v) or pd.isna(b):
                out.append('')
            elif idx == mean_adv_idx:
                out.append(f"{v:+.1f}%")
            else:
                adv = v - b
                if b != 0:
                    pct = adv / abs(b) * 100
                    out.append(f"{v:.3f} ({pct:+.1f}%)")
                else:
                    out.append(f"{v:.3f} ({adv:+.3f})")
        disp[col] = out
    disp.loc[mean_adv_idx, baseline_col] = '+0.0%'

    def _highlight(df):
        mask = tbl_num.gt(tbl_num[baseline_col], axis=0)
        return pd.DataFrame(np.where(mask, 'font-weight: bold;', ''), index=df.index, columns=df.columns)

    styler: Styler = disp.style.apply(_highlight, axis=None)
    display(styler)

def run_feature_mode_group_tables_only(model_key: str, out_dir_name: str):
    cfg = MODEL_CONFIG[model_key]
    model_label = cfg['label']
    out_dir = Path(out_dir_name)
    out_dir.mkdir(parents=True, exist_ok=True)

    df_ok, baseline_feature_battery = load_mode_group_df(model_key)
    if df_ok.empty:
        print(f"[SKIP] {model_key}: missing result folders/baseline")
        return

    available_sets = set(df_ok['feature_battery'].dropna().unique())
    feature_order = [baseline_feature_battery] + sorted(s for s in available_sets if s != baseline_feature_battery)
    feature_order = [s for s in dict.fromkeys(feature_order) if s in available_sets]
    if not feature_order:
        print(f"[SKIP] No feature batteries available for {model_key}")
        return

    print(f"\n=== {model_label} | Tables only ===")
    sub = df_ok[df_ok['feature_battery'].isin(feature_order)].copy()
    display_col = 'feature_battery'
    feature_order_display = feature_order


    clf = sub[sub['task_type'] == 'classification'].dropna(subset=['test_f1'])
    if not clf.empty:
        tbl = clf.pivot_table(index=['base_dataset', 'label'], columns=display_col, values='test_f1', aggfunc='max').reindex(columns=feature_order_display)
        if not tbl.empty:
            style_vs_baseline(tbl, baseline_col=baseline_feature_battery)
            # out = out_dir / f'{model_key}_clf.csv'
            # tbl.to_csv(out)
            # print(f"[SAVE] {out}")

    regr = sub[sub['task_type'] == 'regression'].dropna(subset=['test_r2'])
    if not regr.empty:
        tbl = regr.pivot_table(index=['base_dataset', 'label'], columns=display_col, values='test_r2', aggfunc='max').reindex(columns=feature_order_display)
        if not tbl.empty:
            style_vs_baseline(tbl, baseline_col=baseline_feature_battery)
            # out = out_dir / f'{model_key}_regr.csv'
            # tbl.to_csv(out)
            # print(f"[SAVE] {out}")


# logreg


In [ ]:
MODEL = 'logreg'
print(f"MODEL={MODEL!r}")
run_feature_mode_group_tables_only(MODEL, 'clf_results')


# xgboost


In [ ]:
MODEL = 'xgboost'
print(f"MODEL={MODEL!r}")
run_feature_mode_group_tables_only(MODEL, 'clf_results')


# catboost


In [ ]:
MODEL = 'catboost'
print(f"MODEL={MODEL!r}")
run_feature_mode_group_tables_only(MODEL, 'clf_results')
